In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType,
)
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("AmusementPark").getOrCreate()

rides = spark.createDataFrame(
    [
        ("r1", "Roller Coaster", "Thrill", 24),
        ("r2", "Ferris Wheel", "Observation", 60),
        ("r3", "Log Flume", "Water", 16),
        ("r4", "Bumper Cars", "Family", 20),
        ("r5", "Merry-Go-Round", "Classic", 40),
    ],
    ["ride_id", "ride_name", "type", "capacity"],
)

visitors = spark.createDataFrame(
    [
        ("v1", "r1", "2023-07-01 10:00:00", 5),
        ("v2", "r1", "2023-07-01 10:30:00", 4),
        ("v1", "r2", "2023-07-01 11:00:00", 3),
        ("v3", "r3", "2023-07-01 11:30:00", 2),
        ("v4", "r4", "2023-07-01 12:00:00", 5),
    ],
    ["visitor_id", "ride_id", "timestamp", "rating"],
)

visitors = visitors.withColumn("timestamp", visitors.timestamp.cast(TimestampType()))

rides.show()
visitors.show()

+-------+--------------+-----------+--------+
|ride_id|     ride_name|       type|capacity|
+-------+--------------+-----------+--------+
|     r1|Roller Coaster|     Thrill|      24|
|     r2|  Ferris Wheel|Observation|      60|
|     r3|     Log Flume|      Water|      16|
|     r4|   Bumper Cars|     Family|      20|
|     r5|Merry-Go-Round|    Classic|      40|
+-------+--------------+-----------+--------+

+----------+-------+-------------------+------+
|visitor_id|ride_id|          timestamp|rating|
+----------+-------+-------------------+------+
|        v1|     r1|2023-07-01 10:00:00|     5|
|        v2|     r1|2023-07-01 10:30:00|     4|
|        v1|     r2|2023-07-01 11:00:00|     3|
|        v3|     r3|2023-07-01 11:30:00|     2|
|        v4|     r4|2023-07-01 12:00:00|     5|
+----------+-------+-------------------+------+



In [7]:
rides_avg_rating = visitors.groupBy("ride_id").agg(
    avg("rating").alias("average_rating")
)

# Calculate the overall average rating
overall_avg_rating = rides_avg_rating.agg(avg("average_rating")).collect()[0][0]

# Define the condition for an anomalous ride
condition = (col("average_rating") < overall_avg_rating * 0.5) | (
    col("average_rating") > overall_avg_rating * 1.5
)

# Add the 'is_anomalous' column
rides_avg_rating = rides_avg_rating.withColumn("is_anomalous", condition)

# Join with the rides DataFrame to get the ride_name
result = rides.join(rides_avg_rating, on="ride_id")